<a href="https://colab.research.google.com/github/rdvergara-feua/Vergara-Raynard-CS0065-AN43/blob/main/TA4_VERGARA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Step 1
import pandas as pd

In [2]:
#Step 2
# Part 1A: knowledge base (laptop troubleshooting)
facts = {
    "status": {"powers_on": True, "screen_on": True, "charging": False},
    "hardware": {"battery_pct": 8, "temp_c": 85, "fan_noisy": True},
    "software": {"storage_free_pct": 4, "slow": True},
}

In [3]:
#Step 3
# Show knowledge base
for category, items in facts.items():
    print(category.upper())
    for name, value in items.items():
        print(f"  {name}: {value}")

STATUS
  powers_on: True
  screen_on: True
  charging: False
HARDWARE
  battery_pct: 8
  temp_c: 85
  fan_noisy: True
SOFTWARE
  storage_free_pct: 4
  slow: True


In [4]:
#Step 4
# Part 2A: IF-THEN rules
def rule_no_power(f):
    if not f["powers_on"]:
        return "Check power cable and charger"


def rule_no_display(f):
    if f["powers_on"] and not f["screen_on"]:
        return "Check display cable or connect external monitor"


def rule_low_battery(f):
    if f["battery_pct"] < 15 and not f["charging"]:
        return "Plug in charger"


def rule_overheat(f):
    if f["temp_c"] > 80:
        return "Close heavy apps and clean vents"


def rule_fan(f):
    if f["fan_noisy"] and f["temp_c"] > 70:
        return "Check fan for dust"


def rule_storage(f):
    if f["storage_free_pct"] < 10:
        return "Free up disk space"


def rule_slow(f):
    if f["slow"] and f["storage_free_pct"] >= 10:
        return "Restart and run malware scan"


rules = [rule_no_power, rule_no_display, rule_low_battery, rule_overheat,
         rule_fan, rule_storage, rule_slow]

In [5]:
#Step 5
# Part 2B: inference engine
def flatten(kb):
    return {k: v for cat in kb.values() for k, v in cat.items()}


def infer(kb, rules):
    memory = flatten(kb)
    actions = []
    for rule in rules:
        result = rule(memory)
        if result:
            actions.append((rule.__name__, result))
    return actions

In [6]:
#Step 6
# Part 2C: RBR output
print("Input facts:", flatten(facts))
print()
print("Rule-Based Reasoning Actions:")
for name, action in infer(facts, rules):
    print(f"- {action}  ({name})")

Input facts: {'powers_on': True, 'screen_on': True, 'charging': False, 'battery_pct': 8, 'temp_c': 85, 'fan_noisy': True, 'storage_free_pct': 4, 'slow': True}

Rule-Based Reasoning Actions:
- Plug in charger  (rule_low_battery)
- Close heavy apps and clean vents  (rule_overheat)
- Check fan for dust  (rule_fan)
- Free up disk space  (rule_storage)


In [7]:
#Step 7
# Part 3A: case base
case_base = [
    {"problem": {"powers_on": False, "screen_on": False, "battery_pct": 0, "temp_c": 30,
                 "fan_noisy": False, "storage_free_pct": 50, "slow": False},
     "solution": "Check power cable and charger"},
    {"problem": {"powers_on": True, "screen_on": True, "battery_pct": 50, "temp_c": 90,
                 "fan_noisy": True, "storage_free_pct": 40, "slow": True},
     "solution": "Clean vents and replace thermal paste"},
    {"problem": {"powers_on": True, "screen_on": True, "battery_pct": 70, "temp_c": 45,
                 "fan_noisy": False, "storage_free_pct": 3, "slow": True},
     "solution": "Free up disk space and remove startup apps"},
    {"problem": {"powers_on": True, "screen_on": False, "battery_pct": 60, "temp_c": 40,
                 "fan_noisy": False, "storage_free_pct": 40, "slow": False},
     "solution": "Check display cable or connect external monitor"},
    {"problem": {"powers_on": True, "screen_on": True, "battery_pct": 8, "temp_c": 40,
                 "fan_noisy": False, "storage_free_pct": 40, "slow": False},
     "solution": "Plug in charger and replace battery if it drains fast"},
    {"problem": {"powers_on": True, "screen_on": True, "battery_pct": 60, "temp_c": 45,
                 "fan_noisy": False, "storage_free_pct": 50, "slow": True},
     "solution": "Restart and run malware scan"},
]

print("Cases stored:", len(case_base))
pd.DataFrame([{**c["problem"], "solution": c["solution"]} for c in case_base])

Cases stored: 6


,powers_on,screen_on,battery_pct,temp_c,fan_noisy,storage_free_pct,slow,solution
0,False,False,0,30,False,50,False,Check power cable and charger
1,True,True,50,90,True,40,True,Clean vents and replace thermal paste
2,True,True,70,45,False,3,True,Free up disk space and remove startup apps
3,True,False,60,40,False,40,False,Check display cable or connect external monitor
4,True,True,8,40,False,40,False,Plug in charger and replace battery if it drai...
5,True,True,60,45,False,50,True,Restart and run malware scan


In [8]:
#Step 8
# Part 3B: similarity
RANGES = {"battery_pct": 100, "temp_c": 100, "storage_free_pct": 100}


def similarity(p1, p2):
    scores = []
    for k in p1:
        if k in RANGES:
            scores.append(1 - abs(p1[k] - p2[k]) / RANGES[k])
        else:
            scores.append(1.0 if p1[k] == p2[k] else 0.0)
    return sum(scores) / len(scores)


def retrieve(new, cb, k=3):
    ranked = sorted(((similarity(new, c["problem"]), i) for i, c in enumerate(cb)), reverse=True)
    return ranked[:k]

In [9]:
#Step 9
# Part 3C: reuse
def reuse(new, cb):
    sim, i = retrieve(new, cb, 1)[0]
    return sim, cb[i]["solution"], i

In [10]:
#Step 10
# Part 3D: revise
def revise(solution):
    print("Suggested solution:", solution)
    manual = input("Revised solution (press Enter to keep): ").strip()
    return manual if manual else solution

In [11]:
#Step 11
# Part 3E: retain
def retain(new, solution, cb):
    cb.append({"problem": new, "solution": solution})
    return cb

In [12]:
#Step 12
# Run CBR cycle
new_problem = {"powers_on": True, "screen_on": True, "battery_pct": 55, "temp_c": 88,
               "fan_noisy": True, "storage_free_pct": 35, "slow": True}
print("New problem:", new_problem)

print("\nRETRIEVE (top 3)")
for sim, i in retrieve(new_problem, case_base):
    print(f"  case {i + 1}: similarity {sim:.3f} -> {case_base[i]['solution']}")

print("\nREUSE")
sim, solution, i = reuse(new_problem, case_base)
print(f"  Best match: case {i + 1} (similarity {sim:.3f})")

print("\nREVISE")
final_solution = revise(solution)

print("\nRETAIN")
print("  Cases before:", len(case_base))
case_base = retain(new_problem, final_solution, case_base)
print("  Cases after :", len(case_base))

New problem: {'powers_on': True, 'screen_on': True, 'battery_pct': 55, 'temp_c': 88, 'fan_noisy': True, 'storage_free_pct': 35, 'slow': True}

RETRIEVE (top 3)
  case 2: similarity 0.983 -> Clean vents and replace thermal paste
  case 6: similarity 0.767 -> Restart and run malware scan
  case 3: similarity 0.729 -> Free up disk space and remove startup apps

REUSE
  Best match: case 2 (similarity 0.983)

REVISE
Suggested solution: Clean vents and replace thermal paste
Revised solution (press Enter to keep): 

RETAIN
  Cases before: 6
  Cases after : 7


In [13]:
#Step 13
# Check learning
sim, solution, i = reuse(new_problem, case_base)
print(f"Same problem again -> case {i + 1}, similarity {sim:.3f}")
print("Solution:", solution)

Same problem again -> case 7, similarity 1.000
Solution: Clean vents and replace thermal paste


In [14]:
#Step 14
# Compare RBR vs CBR
scenarios = {
    "A: Known pattern": {"powers_on": True, "screen_on": True, "charging": False, "battery_pct": 8,
                         "temp_c": 85, "fan_noisy": True, "storage_free_pct": 40, "slow": False},
    "B: Near thresholds": {"powers_on": True, "screen_on": True, "charging": False, "battery_pct": 16,
                           "temp_c": 79, "fan_noisy": False, "storage_free_pct": 11, "slow": False},
    "C: Mixed symptoms": {"powers_on": True, "screen_on": True, "charging": True, "battery_pct": 90,
                          "temp_c": 72, "fan_noisy": False, "storage_free_pct": 12, "slow": True},
}

rows = []
for name, s in scenarios.items():
    rbr = [a for _, a in infer({"all": s}, rules)]
    problem = {k: v for k, v in s.items() if k != "charging"}
    sim, sol, i = reuse(problem, case_base)
    rows.append([name, "; ".join(rbr) if rbr else "No rule fired", f"{sol} ({sim:.2f})"])

pd.DataFrame(rows, columns=["scenario", "RBR result", "CBR result (similarity)"])

,scenario,RBR result,CBR result (similarity)
0,A: Known pattern,Plug in charger; Close heavy apps and clean ve...,Plug in charger and replace battery if it drai...
1,B: Near thresholds,No rule fired,Plug in charger and replace battery if it drai...
2,C: Mixed symptoms,Restart and run malware scan,Free up disk space and remove startup apps (0.92)
